# DMOSpeech 2: Decision Handover Pilot (EXP-023)

**Idea.** The teacher's first estimate $T_0$ (Step 0) is, in theory, the same for every random seed: the average output for this text and speaker, including whatever the prompt's recording conditions add. Everything the teacher adds after that, $T_k - T_0$, is this seed's **decision**, which is the pitch variety the student lacks. So the handover passes only the decision, on top of the student's own one-pass draft $S$:

$$x_k = S + \mathrm{slow}(T_k - T_0)$$

Anything the teacher shares across seeds cancels in the difference. **No model is trained: the released teacher and student are used unchanged.**

| Output | What it is | Passes |
|---|---|---:|
| DH_2, DH_3 | the decision at Step 2 or 3, guided teacher | 10, 12 |
| DHc_2, DHc_3 | the same, with the student's own prosody down-weighted (variance-preserving) | 10, 12 |
| DHu_2, DHu_3 | the decision from a teacher without guidance (1 pass per step) | 7, 8 |
| TA, Two-Axis Handover (second candidate) | the prompt SNR decides what crosses (the whole estimate, Voice-Swap, or nothing: the student alone); on the whole-estimate branch the teacher's movement by Step 3 decides when (Step 3, else Step 4) | 11 or 13, 14, 4 |
| ARH, Agreement-Routed Handover (third candidate) | what crosses is decided by whether the teacher and the student agree on the voice at Step 0 (no SNR estimator): agree -> the whole estimate at Step 4, a small gap -> Voice-Swap, a large gap -> the student alone | 14, 14, 6 |

For comparison: 0.25 reference 21, Voice-Swap 14.

**What runs.**
- **Quality:** the six outputs on 20 development speakers x clean, white noise 20 dB and 15 dB (60 cases), scored against the stored reference.
- **Assumption check:** the teacher alone with 4 seeds on 10 speakers x clean and noisy prompts. Is $T_0$ really the same for every seed?
- **Pitch variety:** CVf0 of the six outputs, and of the Two-Axis Handover's Step-3 and Step-4 handovers and its per-seed choice between them, on 10 speakers x 20 seeds. The Two-Axis Handover already passed every quality rule on stored outputs; its pitch variety at Step 3 is the one rule never measured (G3).
- **Agreement-Routed Handover:** no extra synthesis. The quality cases already record the Step-0 voice gap, so the analysis replays its routing out of fold and compares it with the prompt-SNR rule on the same cases (G4).

Stored outputs of earlier runs on the same cases and seeds are reused for comparison.

**Time.** About 1.5 to 2 hours on a T4 (an estimate, not a measurement). Results are saved to Google Drive after every case, so a disconnected run resumes.

**Status.** A development pilot: it decides whether a frozen test on new speakers is worth running, and cannot support a claim. The go/no-go rules are frozen in `preregistration.json` (amended on 2026-11-13, before any synthesis, to add the Two-Axis and Agreement-Routed Handovers). The runner refuses to start if any of its code, the preregistration or the stored input files have changed, if the GPU is not a T4, or if the smoke test did not reproduce the stored scores.

---
### Instructions
1. Set the runtime to **T4 GPU** (`Runtime > Change runtime type`).
2. **Runtime > Run all.** The smoke test must pass before the full run can start.
3. If the browser disconnects, reconnect and run **Step 4b**. If the machine resets, re-run Steps 1 to 2b, then Step 4: it continues from the last saved case.
4. When it finishes, Step 6 downloads `DMOSpeech2_EXP023_Decision_Handover_Pilot_Results.zip` (also saved to Drive): send that file.

### Step 1: Check the GPU (Tesla T4 required)

In [ ]:
import torch
assert torch.cuda.is_available(), 'CUDA not available. Switch the runtime to T4 GPU.'
gpu_name = torch.cuda.get_device_name(0)
print('GPU:', gpu_name)
assert 'T4' in gpu_name, f'Expected a Tesla T4 (found {gpu_name}). Student noise depends on the GPU, so the stored labels would not reproduce.'

### Step 2: Clone the Repositories and Install Dependencies

In [ ]:
import os, subprocess, hashlib, sys
THESIS_BRANCH = 'rhnl-3'
token = ''
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN') or ''
except Exception:
    pass
if not os.path.exists('/content/Final-Thesis-Setup/.git'):
    subprocess.run(['git', 'clone', '-b', THESIS_BRANCH, f"https://{token + '@' if token else ''}github.com/Shonget-Thesis/Final-Thesis-Setup.git", '/content/Final-Thesis-Setup'], check=True)
else:
    subprocess.run(['git', '-C', '/content/Final-Thesis-Setup', 'fetch', 'origin', THESIS_BRANCH], check=True)
    subprocess.run(['git', '-C', '/content/Final-Thesis-Setup', 'checkout', '-B', THESIS_BRANCH, f'origin/{THESIS_BRANCH}'], check=True)

for needed in ('code/colab/run_decision_handover_pilot.py', 'research/experiments/EXP-023-decision-handover-pilot/preregistration.json',
               'research/experiments/EXP-022-slow-part-handover-prelim/results/slowpart.csv'):
    if not os.path.exists(f'/content/Final-Thesis-Setup/{needed}'):
        raise FileNotFoundError(f'{needed} is not on branch {THESIS_BRANCH}. Push it to GitHub first.')

DMO_COMMIT = '8a7868fc3345702a586173447f4869205b501f26'
if not os.path.exists('/content/Shonget-DMOSpeech2'):
    !git clone https://github.com/Shonget-Thesis/Shonget-DMOSpeech2.git /content/Shonget-DMOSpeech2
%cd /content/Shonget-DMOSpeech2
!git fetch -q origin && git checkout -q {DMO_COMMIT}

!pip install -q "numpy==2.2.3" "pandas==2.2.3" audioop-lts
!pip install -q -r requirements.txt
!pip install -q torchdiffeq vocos speechbrain transformers jiwer soundfile librosa pyarrow scipy jieba accelerate huggingface_hub
print('Code and packages ready.')

### Step 2b: Load the Models and Dataset (Saved in Google Drive)
The first run downloads the checkpoints and dataset, then saves a copy to `MyDrive/dmospeech2_cache/`. Later runs copy them from there, so nothing is downloaded again. The checkpoint SHA-256 hashes are checked every time, and a damaged copy is replaced automatically. If Drive is full, the run continues without saving a copy.

In [ ]:
# Models and dataset are kept once in Google Drive and reused by later runs.
import os, shutil, tarfile, hashlib, subprocess
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
os.chdir('/content/Shonget-DMOSpeech2')

CACHE = '/content/drive/MyDrive/dmospeech2_cache'
TMP_TAR = '/content/seed-tts-eval-en.tar'
os.makedirs(f'{CACHE}/ckpts', exist_ok=True)
os.makedirs('ckpts', exist_ok=True)
BASE_URL = 'https://huggingface.co/yl4579/DMOSpeech2/resolve/main'
CKPTS = {'model_85000.pt': 'e143188f7c46b69abb60786937d79b89dac9777d3c4c1e91a97634bcd39ebf2c',
         'model_1500.pt': '6bd50428f3b5f0f25c1a60a0053f40ee47c817bda03c8ee120100f22680ae4c2'}


def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(1 << 24), b''):
            h.update(block)
    return h.hexdigest()


def save_to_cache(local, cached):
    # The cache is optional: a full Drive must not stop the experiment.
    try:
        shutil.copy(local, cached + '.tmp')
        os.replace(cached + '.tmp', cached)
        print(f'Saved to the Drive cache: {os.path.basename(cached)}')
    except OSError as e:
        print(f'Could not save {os.path.basename(cached)} to Drive ({e}). Continuing without the cache.')
        if os.path.exists(cached + '.tmp'):
            os.remove(cached + '.tmp')


for name, digest in CKPTS.items():
    local, cached = f'ckpts/{name}', f'{CACHE}/ckpts/{name}'
    ok = os.path.exists(local) and sha256(local) == digest
    if not ok and os.path.exists(cached):
        print(f'{name}: copying from the Drive cache')
        shutil.copy(cached, local)
        ok = sha256(local) == digest
        if not ok:
            print(f'{name}: the Drive copy is damaged, so it will be downloaded again')
            os.remove(cached)
            os.remove(local)
    if not ok:
        print(f'{name}: downloading')
        subprocess.run(['wget', '-q', '-c', f'{BASE_URL}/{name}', '-O', local], check=True)
        assert sha256(local) == digest, f'{name}: SHA-256 mismatch (delete ckpts/{name} and re-run this step)'
    print(f'{name}: SHA-256 verified')
    if not os.path.exists(cached):
        save_to_cache(local, cached)

DATA, ARCHIVE = 'data/seed-tts-eval', f'{CACHE}/seed-tts-eval-en.tar'
PROBE = f'{DATA}/en/prompt-wavs/common_voice_en_10933823.wav'


def have_data():
    return os.path.exists(PROBE) and os.path.getsize(PROBE) >= 1000 and os.path.exists(f'{DATA}/en/meta.lst')


if not have_data() and os.path.exists(ARCHIVE):
    print('Dataset: unpacking from the Drive cache')
    shutil.rmtree(DATA, ignore_errors=True)
    os.makedirs('data', exist_ok=True)
    try:
        with tarfile.open(ARCHIVE) as tar:
            tar.extractall('data')
    except (tarfile.TarError, OSError, EOFError) as e:
        print(f'The Drive copy of the dataset could not be read ({e}).')
    if not have_data():
        print('Dataset: the Drive copy is damaged, so it will be downloaded again')
        os.remove(ARCHIVE)
if not have_data():
    print('Dataset: downloading')
    shutil.rmtree(DATA, ignore_errors=True)
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id='zhaochenyang20/seed-tts-eval', repo_type='dataset', local_dir=DATA)
    assert have_data(), 'The dataset download is incomplete. Re-run this step.'
if not os.path.exists(ARCHIVE):
    print('Dataset: saving one copy to the Drive cache')
    with tarfile.open(TMP_TAR, 'w') as tar:
        tar.add(f'{DATA}/en', arcname='seed-tts-eval/en')
    save_to_cache(TMP_TAR, ARCHIVE)
    os.remove(TMP_TAR)
print('Models and dataset ready.')

### Step 3: Smoke Test (2 speakers, a few minutes)
Runs every output for 2 development speakers (clean and Gaussian 20 dB), one short seed check and one short pitch-variety pair. It checks two things:
- **Stored scores:** this runner reproduces the stored Slow-Part scores at Step 3 (EXP-022).
- **Teacher loop:** its own teacher loop matches the released guided teacher exactly. The unguided loop is the same loop without the second pass.

**If either check fails, it stops here and the full run will refuse to start.** Writes to a separate `smoke/` folder.

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, sys, subprocess
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'{THESIS}/code/colab/run_decision_handover_pilot.py'
PERSIST = '/content/drive/MyDrive/dmospeech2_results'
OUT = f'{PERSIST}/exp023_decision_handover_pilot'
LOG = f'{PERSIST}/exp023_decision_handover_pilot_log.txt'
os.makedirs(OUT, exist_ok=True)
os.chdir(DMO)
print(f'Results in Google Drive: {OUT}')

proc = subprocess.Popen([sys.executable, '-u', RUNNER, 'smoke', '--out', OUT, '--dmo_root', DMO],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
for line in proc.stdout:
    print(line, end='', flush=True)
assert proc.wait() == 0, f'Smoke test failed with code {proc.returncode}'

import pandas as pd
print(pd.read_csv(f'{OUT}/smoke/pilot.csv')[['utt_id', 'condition', 'prompt_snr', 'dc_g_2', 'dr_g_2', 'vg_g', 'wer_DH_2', 'wer_DHu_3', 'wer_STS_3', 'similarity_DH_3', 'teacher_loop_max_abs_diff', 'case_seconds']].to_string())

### Step 4: Full Run + Analysis (about 1.5 to 2 hours, estimated)
Runs in the background and writes its output to `MyDrive/dmospeech2_results/exp023_decision_handover_pilot_log.txt`; this cell prints one progress line per minute. If only the browser disconnects, the job keeps running: reconnect and run **Step 4b**. If the machine resets, re-run Steps 1 to 2b, then this cell.

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, sys, subprocess
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'{THESIS}/code/colab/run_decision_handover_pilot.py'
PERSIST = '/content/drive/MyDrive/dmospeech2_results'
OUT = f'{PERSIST}/exp023_decision_handover_pilot'
LOG = f'{PERSIST}/exp023_decision_handover_pilot_log.txt'
os.makedirs(OUT, exist_ok=True)
os.chdir(DMO)
print(f'Results in Google Drive: {OUT}')

import time, re
cmd = f'{sys.executable} -u "{RUNNER}" run --out "{OUT}" --dmo_root {DMO}'
running = subprocess.run(['pgrep', '-f', 'run_decision_handover_pilot.py'], capture_output=True, text=True).stdout.split()
if running:
    pid = int(running[0])
    alive = lambda: os.path.exists(f'/proc/{pid}')
    print(f'A job is already running (pid {pid}); monitoring it instead of starting a new one.')
else:
    proc = subprocess.Popen(f'nohup {cmd} >> "{LOG}" 2>&1', shell=True)
    alive = lambda: proc.poll() is None
    print(f'Started (pid {proc.pid}). Full log: {LOG}')

last = ''
while alive():
    time.sleep(60)
    try:
        lines = open(LOG, encoding='utf-8', errors='ignore').read().splitlines()
        prog = [l for l in lines if re.match(r'^\[\d+/\d+\]', l) or l.startswith('[OK]') or 'Error' in l or 'SystemExit' in l]
        if prog and prog[-1] != last:
            last = prog[-1]
            print(time.strftime('%H:%M'), last)
    except FileNotFoundError:
        pass
print(f"Finished{'' if running else f' with exit code {proc.returncode}'}. Last lines of the log:")
for line in open(LOG, encoding='utf-8', errors='ignore').read().splitlines()[-15:]:
    print(line)

### Step 4b: Check Progress After a Reconnect
Shows whether the background job is running and how many cases are saved. Does **not** start a second job.

In [ ]:
import os, re, subprocess
PERSIST = '/content/drive/MyDrive/dmospeech2_results'
OUT = f'{PERSIST}/exp023_decision_handover_pilot'
LOG = f'{PERSIST}/exp023_decision_handover_pilot_log.txt'
running = subprocess.run(['pgrep', '-f', 'run_decision_handover_pilot.py'], capture_output=True, text=True).stdout.split()
print('Job running:', 'yes (pid ' + ', '.join(running) + ')' if running else 'no')
if os.path.exists(LOG):
    lines = open(LOG, encoding='utf-8', errors='ignore').read().splitlines()
    prog = [l for l in lines if re.match(r'^\[\d+/\d+\]', l) or l.startswith('[OK]') or 'Error' in l]
    print('Latest progress:', prog[-1] if prog else '(none yet)')
for name, total in (('pilot.csv', 60), ('seeds.csv', 20), ('diversity.csv', 10)):
    path = os.path.join(OUT, name)
    n = sum(1 for _ in open(path, encoding='utf-8')) - 1 if os.path.exists(path) else 0
    print(f'{name}: {n} of {total} saved')

### Step 5: View the Report
If the run stopped early, add `'--allow_partial'` to the command for an interim look (the report is then marked PARTIAL).

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, sys, subprocess
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'{THESIS}/code/colab/run_decision_handover_pilot.py'
PERSIST = '/content/drive/MyDrive/dmospeech2_results'
OUT = f'{PERSIST}/exp023_decision_handover_pilot'
LOG = f'{PERSIST}/exp023_decision_handover_pilot_log.txt'
os.makedirs(OUT, exist_ok=True)
os.chdir(DMO)
print(f'Results in Google Drive: {OUT}')

from IPython.display import Markdown, display
rep = f'{OUT}/PILOT_REPORT.md'
if not os.path.exists(rep):
    subprocess.run([sys.executable, RUNNER, 'analyze', '--out', OUT], check=True)
display(Markdown(open(rep, encoding='utf-8').read()))

### Step 6: Zip All Outputs and Download
Packs the CSV files, preregistration, results JSON, report and runner scripts into one ZIP, also saved to Drive. Audio is not included.

In [ ]:
# Self-contained setup: survives a session restart or disconnect.
# Results are written straight to Google Drive, so a re-run resumes where it stopped.
import os, sys, subprocess
from google.colab import drive
if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'
DMO = '/content/Shonget-DMOSpeech2'
THESIS = '/content/Final-Thesis-Setup'
RUNNER = f'{THESIS}/code/colab/run_decision_handover_pilot.py'
PERSIST = '/content/drive/MyDrive/dmospeech2_results'
OUT = f'{PERSIST}/exp023_decision_handover_pilot'
LOG = f'{PERSIST}/exp023_decision_handover_pilot_log.txt'
os.makedirs(OUT, exist_ok=True)
os.chdir(DMO)
print(f'Results in Google Drive: {OUT}')

import zipfile, datetime, torch
zip_name = 'DMOSpeech2_EXP023_Decision_Handover_Pilot_Results.zip'
git = lambda *a: subprocess.run(['git', *a, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
with open(f'{OUT}/RUN_INFO.txt', 'w') as f:
    f.write(f"dmospeech2_commit: {git()}\n")
    f.write(f"thesis_commit: {git('-C', THESIS)}\n")
    f.write(f"zipped_at_utc: {datetime.datetime.now(datetime.timezone.utc).isoformat()}\n")
    f.write(f"device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}\n")
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for base, dirs, names in os.walk(OUT):
        for fname in names:
            if fname.startswith('tmp_') or fname.endswith('.wav'):
                continue
            full = os.path.join(base, fname)
            zf.write(full, os.path.relpath(full, OUT))
    for script in ('run_decision_handover_pilot.py', 'run_selective_handover_prelim.py', 'run_grid32_prelim.py', 'run_noise_guidance_prelim.py',
                   'run_endpoint_extrapolation_prelim.py', 'run_consistency_gate_prelim.py', 'run_followup_tests.py'):
        zf.write(f'{THESIS}/code/colab/{script}', f'code/{script}')
print(f'Created {zip_name} ({os.path.getsize(zip_name) / 1e6:.2f} MB)')
!cp "{zip_name}" "{PERSIST}/"
print(f'Also saved to Google Drive: {PERSIST}/{zip_name}')
try:
    from google.colab import files
    files.download(zip_name)
except Exception:
    print(f'Download manually from the Files panel: {DMO}/{zip_name}')